# 14 - Channel Contribution Analysis

## Objective

Convert model outputs into business insights by estimating:

- Base Sales
- Incremental Sales
- Channel Contribution
- Contribution %
- ROI
- ROAS

> **Note:** True production MMM decomposes contributions using fitted transformed media variables and model coefficients. This notebook demonstrates the workflow using the trained feature matrix.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import Ridge

ROOT=Path.cwd()
DATA=ROOT/"data"/"processed"/"marketing_mix_model_ready.csv"

df=pd.read_csv(DATA)

target="Sales"
X=df.drop(columns=[target])
y=df[target]

model=Ridge(alpha=1.0)
model.fit(X,y)

coefs=pd.Series(model.coef_,index=X.columns)
intercept=model.intercept_

pred=model.predict(X)
df["Predicted_Sales"]=pred


## 1. Base vs Incremental Sales

In [ ]:

base_sales=np.repeat(intercept,len(df))
incremental=pred-base_sales

df["Base_Sales"]=base_sales
df["Incremental_Sales"]=incremental

summary=pd.DataFrame({
    "Metric":["Actual Sales","Predicted Sales","Base Sales","Incremental Sales"],
    "Value":[df["Sales"].sum(),pred.sum(),base_sales.sum(),incremental.sum()]
})
display(summary)


## 2. Channel Contribution

In [ ]:

channels=[c for c in X.columns if "Google" in c or "Meta" in c or "TV" in c or "Email" in c or "Radio" in c]

rows=[]
for ch in channels:
    contrib=(X[ch]*coefs[ch]).sum()
    rows.append([ch,contrib])

contrib_df=pd.DataFrame(rows,columns=["Channel","Contribution"])
contrib_df["Contribution_%"]=100*contrib_df["Contribution"]/contrib_df["Contribution"].sum()
display(contrib_df.sort_values("Contribution",ascending=False))


## 3. Contribution Chart

In [ ]:

plot_df=contrib_df.sort_values("Contribution")
plt.figure(figsize=(10,5))
plt.barh(plot_df["Channel"],plot_df["Contribution"])
plt.title("Estimated Channel Contribution")
plt.tight_layout()
plt.show()


## 4. ROI & ROAS

In [ ]:

spend_cols=[c for c in X.columns if c.endswith("_Hill")]
roi=[]

for col in spend_cols:
    spend=df[col].sum()
    if col.replace("_Hill","")+"_Hill" in coefs.index:
        revenue=max((df[col]*coefs[col]).sum(),0)
    else:
        revenue=0
    roi.append([col,spend,revenue,
                revenue/(spend+1e-9)])

roi_df=pd.DataFrame(roi,columns=[
    "Channel","Total Spend Feature","Estimated Revenue","ROAS"
])

display(roi_df.sort_values("ROAS",ascending=False))


## 5. Waterfall-style Table

In [ ]:

waterfall=contrib_df.copy()
waterfall.loc[len(waterfall)] = ["Base Sales",base_sales.sum(),100-base_sales.sum()*0]
display(waterfall)


## 6. Marketing Recommendations

In [ ]:

top=contrib_df.sort_values("Contribution",ascending=False).head(3)

recommendations=pd.DataFrame({
    "Recommendation":[
        "Increase investment in strongest channels",
        "Review weak channels",
        "Validate low-contributing campaigns",
        "Run budget optimization"
    ]
})

display(top)
display(recommendations)


# Executive Summary

This notebook translates regression outputs into business language.

Typical executive questions answered:

- Which channels drive the most sales?
- How much of sales is baseline demand?
- Which channels deserve additional budget?
- Which channels should be optimized?

## Interview Questions

1. What is the difference between base and incremental sales?
2. Why is contribution analysis more useful than coefficients alone?
3. How does ROAS differ from ROI?
4. Why should contribution estimates be interpreted with business context?
5. What limitations exist in simple coefficient-based decomposition?

## Next Notebook

**15_ROI_ROAS_Analysis.ipynb**

We'll perform a deeper financial analysis including:
- Channel efficiency
- Cost per incremental sale
- Marginal ROI
- Spend efficiency curves
- Executive KPI dashboard
